In [1]:
import mne
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from moabb.datasets import BNCI2014_001
import plotly.io as pio

pio.renderers.default = "vscode"




In [2]:
# 2014
participant = 1
dataset = BNCI2014_001()
data = dataset.get_data(subjects=[participant])

session_key = list(data[participant].keys())[0]
run_key = list(data[participant][session_key].keys())[0]
raw = data[participant][session_key][run_key]

events, event_id = mne.events_from_annotations(raw)
epochs = mne.Epochs(raw, events, event_id=event_id,
tmin=-0.2, tmax=2.0, baseline=None, preload=True)

classes = list(event_id.keys())
classe = classes[3]
idx = 0

ep_classe = epochs[classe]
trial = ep_classe[idx].get_data()[0]
times = ep_classe.times
fig = go.Figure()

for ch_idx, ch_name in enumerate(epochs.ch_names):
    fig.add_trace(go.Scatter(x=times, y=trial[ch_idx], mode='lines', name=ch_name))

fig.update_layout(
    title=f"Sujeito {participant} | Classe: {classe} | Trial {idx+1}/{len(ep_classe)}",
    xaxis_title="Tempo (s)",
    yaxis_title="Amplitude (µV)",
    margin=dict(l=10, r=10, t=40, b=10),
    legend_title="Canal"
)
display(fig)





Used Annotations descriptions: [np.str_('feet'), np.str_('left_hand'), np.str_('right_hand'), np.str_('tongue')]
Not setting metadata
48 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 48 events and 551 original time points ...
0 bad epochs dropped


In [3]:
# SELECIONAR CLASSE

participant = 1
dataset = BNCI2014_001() # dataset usado
data = dataset.get_data(subjects=[participant]) 

session_key = list(data[participant].keys())[0]
run_key = list(data[participant][session_key].keys())[0]
raw = data[participant][session_key][run_key]

events, event_id = mne.events_from_annotations(raw) 
epochs = mne.Epochs(raw, events, event_id=event_id,
tmin=-0.2, tmax=2.0, baseline=None, preload=True)

epochs.pick('eeg') # mantem apenas os canais de eeg

classes = list(event_id.keys()) 
idx = 0
ch_names = epochs.ch_names
n_ch = len(ch_names)

fig = go.Figure()
traces_por_classes = []

for c_idx, classe in enumerate(classes):

    ep_classe = epochs[classe] 
    trial = ep_classe[idx].get_data()[0]
    times = ep_classe.times

    inicio = len(fig.data)
    for ch_idx, ch_name in enumerate(ch_names):
        fig.add_trace(go.Scatter(x=times, y=trial[ch_idx], mode='lines', name=ch_name, legendgroup=f"classe_{c_idx}", visible=(c_idx == 0)))
    traces_por_classes.append((inicio, inicio + n_ch)) 

total_traces = len(fig.data)
buttons = []

for c_idx, classe in enumerate(classes):
    visible_mask = [False] * total_traces
    inicio, fim = traces_por_classes[c_idx]
    for i in range(inicio, fim):
        visible_mask[i] = True

    buttons.append(dict(
        label=f"Classe {classe}",
        method="update",
        args=[
            {"visible": visible_mask},
            {"title": f"Sujeito{participant} | Classe: {classe} | Trial {idx+1}"}
        ]
    ))

fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=0.3,
            xanchor="left",
            y=1.15,
            yanchor="top"
        ),
    ],
    title=f"Sujeito {participant} | Classe: {classes[0]} | Trial {idx+1}",
    xaxis_title="Tempo (s)",
    yaxis_title="Amplitude (µV)",
    margin=dict(l=10, r=10, t=60, b=10),
    legend_title="Canal"
)
display(fig)





Used Annotations descriptions: [np.str_('feet'), np.str_('left_hand'), np.str_('right_hand'), np.str_('tongue')]
Not setting metadata
48 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 48 events and 551 original time points ...
0 bad epochs dropped


In [ ]:
# SELECIONAR SESSÃO

participant = 1
dataset = BNCI2014_001()
data = dataset.get_data(subjects=[participant])

session_keys = list(data[participant].keys())

classe_alvo = None   
idx = 0               # índice do trial dentro de cada sessão
ch_names = None
n_ch = None

fig = go.Figure()
traces_por_sessao = []

for s_idx, session_key in enumerate(session_keys):
    # pega o primeiro run dessa sessão
    run_key = list(data[participant][session_key].keys())[0]
    raw = data[participant][session_key][run_key]

    events, event_id = mne.events_from_annotations(raw)
    epochs = mne.Epochs(raw, events, event_id=event_id,
                         tmin=0, tmax=4.0, baseline=None, preload=True)
    epochs.pick('eeg')

    if ch_names is None:
        ch_names = epochs.ch_names
        n_ch = len(ch_names)
        classe_alvo = list(event_id.keys())[0]  

    ep_classe = epochs[classe_alvo]
    trial = ep_classe[idx].get_data()[0]
    times = ep_classe.times

    inicio = len(fig.data)
    for ch_idx, ch_name in enumerate(ch_names):
        fig.add_trace(go.Scatter(
            x=times,
            y=trial[ch_idx],
            mode='lines',
            name=ch_name,
            legendgroup=f"sessao_{s_idx}",
            visible=(s_idx == 0)   
        ))
    traces_por_sessao.append((inicio, inicio + n_ch))


total_traces = len(fig.data)
buttons = []

for s_idx, session_key in enumerate(session_keys):
    visible_mask = [False] * total_traces
    inicio, fim = traces_por_sessao[s_idx]
    for i in range(inicio, fim):
        visible_mask[i] = True

    buttons.append(dict(
        label=f"Sessão {session_key}",
        method="update",
        args=[
            {"visible": visible_mask},
            {"title": f"Sujeito {participant} | Sessão: {session_key} | Classe: {classe_alvo} | Trial {idx+1}"}
        ]
    ))

fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=0.36,
            xanchor="left",
            y=1.15,
            yanchor="top"
        ),
    ],
    title=f"Sujeito {participant} | Sessão: {session_keys[0]} | Classe: {classe_alvo} | Trial {idx+1}",
    xaxis_title="Tempo (s)",
    yaxis_title="Amplitude (µV)",
    margin=dict(l=10, r=10, t=60, b=10),
    legend_title="Canal"
)

display(fig)


Used Annotations descriptions: [np.str_('feet'), np.str_('left_hand'), np.str_('right_hand'), np.str_('tongue')]
Not setting metadata
48 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 48 events and 551 original time points ...
0 bad epochs dropped
Used Annotations descriptions: [np.str_('feet'), np.str_('left_hand'), np.str_('right_hand'), np.str_('tongue')]
Not setting metadata
48 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 48 events and 551 original time points ...
0 bad epochs dropped


In [ ]:
# SELECIONAR TRIAL

participant = 1
dataset = BNCI2014_001()
data = dataset.get_data(subjects=[participant])

session_key = list(data[participant].keys())[0]
run_key = list(data[participant][session_key].keys())[0]
raw = data[participant][session_key][run_key]

events, event_id = mne.events_from_annotations(raw)
epochs = mne.Epochs(raw, events, event_id=event_id,
                     tmin=0, tmax=4, baseline=None, preload=True)
epochs.pick('eeg')


classes = list(event_id.keys())
classe = classes[1] 

ep_classe = epochs[classe]
ch_names = epochs.ch_names
n_ch = len(ch_names)
times = ep_classe.times
n_trials = len(ep_classe)

print(f"Classe '{classe}' tem {n_trials} trials disponíveis (índices 0 a {n_trials-1})")


fig = go.Figure()
traces_por_trial = []

trials_data = ep_classe.get_data()  

for t_idx in range(n_trials):
    trial = trials_data[t_idx]
    inicio = len(fig.data)

    for ch_idx, ch_name in enumerate(ch_names):
        fig.add_trace(go.Scatter(
            x=times,
            y=trial[ch_idx],
            mode='lines',
            name=ch_name,
            legendgroup=f"trial_{t_idx}",
            visible=(t_idx == 0)  
        ))
    traces_por_trial.append((inicio, inicio + n_ch))


total_traces = len(fig.data)
buttons = []

for t_idx in range(n_trials):
    visible_mask = [False] * total_traces
    inicio, fim = traces_por_trial[t_idx]
    for i in range(inicio, fim):
        visible_mask[i] = True

    buttons.append(dict(
        label=f"Trial {t_idx + 1}",
        method="update",
        args=[
            {"visible": visible_mask},
            {"title": f"Sujeito {participant} | Classe: {classe} | Trial {t_idx + 1}/{n_trials}"}
        ]
    ))

fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=0.3,
            xanchor="left",
            y=1.15,
            yanchor="top"
        ),
    ],
    title=f"Sujeito {participant} | Classe: {classe} | Trial 1/{n_trials}",
    xaxis_title="Tempo (s)",
    yaxis_title="Amplitude (µV)",
    margin=dict(l=10, r=10, t=60, b=10),
    legend_title="Canal"
)

display(fig)


Used Annotations descriptions: [np.str_('feet'), np.str_('left_hand'), np.str_('right_hand'), np.str_('tongue')]
Not setting metadata
48 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 48 events and 551 original time points ...
0 bad epochs dropped
Classe 'left_hand' tem 12 trials disponíveis (índices 0 a 11)


In [ ]:
# SELECIONAR PARTICIPANTES

dataset = BNCI2014_001()
data = dataset.get_data() 

participants = list(data.keys())  # lista de IDs dos participantes

classe_alvo = None
idx = 0            # índice do trial usado para todos os participantes
ch_names = None
n_ch = None

fig = go.Figure()
traces_por_participante = []

for p_idx, participant in enumerate(participants):
    # pega a primeira sessão e o primeiro run desse participante
    session_key = list(data[participant].keys())[0]
    run_key = list(data[participant][session_key].keys())[0]
    raw = data[participant][session_key][run_key]

    events, event_id = mne.events_from_annotations(raw)
    epochs = mne.Epochs(raw, events, event_id=event_id,
                         tmin=0, tmax=4, baseline=None, preload=True)
    epochs.pick('eeg')

    if ch_names is None:
        ch_names = epochs.ch_names
        n_ch = len(ch_names)
        classe_alvo = list(event_id.keys())[0]  

    ep_classe = epochs[classe_alvo]
    trial = ep_classe[idx].get_data()[0]
    times = ep_classe.times

    inicio = len(fig.data)
    for ch_idx, ch_name in enumerate(ch_names):
        fig.add_trace(go.Scatter(
            x=times,
            y=trial[ch_idx],
            mode='lines',
            name=ch_name,
            legendgroup=f"participante_{p_idx}",
            visible=(p_idx == 0)  
        ))
    traces_por_participante.append((inicio, inicio + n_ch))


total_traces = len(fig.data)
buttons = []

for p_idx, participant in enumerate(participants):
    visible_mask = [False] * total_traces
    inicio, fim = traces_por_participante[p_idx]
    for i in range(inicio, fim):
        visible_mask[i] = True

    buttons.append(dict(
        label=f"Sujeito {participant}",
        method="update",
        args=[
            {"visible": visible_mask},
            {"title": f"Sujeito {participant} | Classe: {classe_alvo} | Trial {idx+1}"}
        ]
    ))

fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=0.26,
            xanchor="left",
            y=1.15,
            yanchor="top"
        ),
    ],
    title=f"Sujeito {participants[0]} | Classe: {classe_alvo} | Trial {idx+1}",
    xaxis_title="Tempo (s)",
    yaxis_title="Amplitude (µV)",
    margin=dict(l=10, r=10, t=60, b=10),
    legend_title="Canal"
)

fig.show()


Used Annotations descriptions: [np.str_('feet'), np.str_('left_hand'), np.str_('right_hand'), np.str_('tongue')]
Not setting metadata
48 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 48 events and 551 original time points ...
0 bad epochs dropped
Used Annotations descriptions: [np.str_('feet'), np.str_('left_hand'), np.str_('right_hand'), np.str_('tongue')]
Not setting metadata
48 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 48 events and 551 original time points ...
0 bad epochs dropped
Used Annotations descriptions: [np.str_('feet'), np.str_('left_hand'), np.str_('right_hand'), np.str_('tongue')]
Not setting metadata
48 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 48 events and 551 original time points ...
0 bad epochs dropped
Used Annotations descriptions: [np.str_('feet'), 

In [ ]:
# SELECIONAR PARTICIPANTES,TRIAL,SESSÃO,CLASSE
# como o dataset de cada sujeito é separado é possível extrair mais levemente as informações
dataset = BNCI2014_001()
data = dataset.get_data()

participantes = list(data.keys())
cache_epochs = {}  # evita reprocessar mne.Epochs a cada troca

def carregar_epochs(participante, sessao):
    chave = (participante, sessao)
    if chave in cache_epochs:
        return cache_epochs[chave]

    run_key = list(data[participante][sessao].keys())[0]
    raw = data[participante][sessao][run_key]

    events, event_id = mne.events_from_annotations(raw)
    epochs = mne.Epochs(raw, events, event_id=event_id,
                         tmin=0, tmax=4.0, baseline=None, preload=True)
    epochs.pick('eeg')

    cache_epochs[chave] = epochs
    return epochs

# --- dropdowns ---
dd_sujeito = widgets.Dropdown(options=participantes, description='Sujeito:')
dd_sessao  = widgets.Dropdown(description='Sessão:')
dd_classe  = widgets.Dropdown(description='Classe:')
dd_trial   = widgets.Dropdown(description='Trial:')

fig = go.FigureWidget()
fig.update_layout(
    xaxis_title="Tempo (s)",
    yaxis_title="Amplitude (µV)",
    legend_title="Canal",
    margin=dict(l=10, r=10, t=60, b=10),
    width=1300,   
    height=500 
)

ch_names_atual = []

def plotar():
    participante = dd_sujeito.value
    sessao = dd_sessao.value
    classe = dd_classe.value
    trial_num = dd_trial.value
    if None in (participante, sessao, classe, trial_num):
        return

    epochs = carregar_epochs(participante, sessao)
    ep_classe = epochs[classe]
    trial_data = ep_classe[trial_num - 1].get_data()[0]
    times = ep_classe.times

    global ch_names_atual
    with fig.batch_update():
        if ch_names_atual != epochs.ch_names:
            fig.data = []
            for ch in epochs.ch_names:
                fig.add_trace(go.Scatter(x=times, y=[], mode='lines', name=ch))
            ch_names_atual = epochs.ch_names[:]
        for i in range(len(ch_names_atual)):
            fig.data[i].x = times
            fig.data[i].y = trial_data[i]
        fig.layout.title = f"Sujeito {participante} | Sessão {sessao} | Classe: {classe} | Trial {trial_num}"

def atualizar_sessoes(change=None):
    sessoes = list(data[dd_sujeito.value].keys())
    dd_sessao.options = sessoes
    if sessoes:
        dd_sessao.value = sessoes[0]   # <-- força a seleção, não confia no auto-select
    atualizar_classes()

def atualizar_classes(change=None):
    participante, sessao = dd_sujeito.value, dd_sessao.value
    if participante is None or sessao is None:
        return
    epochs = carregar_epochs(participante, sessao)
    classes = list(epochs.event_id.keys())
    dd_classe.options = classes
    if classes:
        dd_classe.value = classes[0]   # <-- idem
    atualizar_trials()

def atualizar_trials(change=None):
    participante, sessao, classe = dd_sujeito.value, dd_sessao.value, dd_classe.value
    if None in (participante, sessao, classe):
        return
    epochs = carregar_epochs(participante, sessao)
    n_trials = len(epochs[classe])
    trials = list(range(1, n_trials + 1))
    dd_trial.options = trials
    if trials:
        dd_trial.value = trials[0]     # <-- idem
    plotar()

dd_sujeito.observe(atualizar_sessoes, names='value')
dd_sessao.observe(atualizar_classes, names='value')
dd_classe.observe(atualizar_trials, names='value')
dd_trial.observe(lambda change: plotar(), names='value')

atualizar_sessoes()

controles = widgets.HBox([dd_sujeito, dd_sessao, dd_classe, dd_trial])
display(controles, fig)

Used Annotations descriptions: [np.str_('feet'), np.str_('left_hand'), np.str_('right_hand'), np.str_('tongue')]
Not setting metadata
48 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 48 events and 551 original time points ...
0 bad epochs dropped


FigureWidget({
    'data': [{'mode': 'lines',
              'name': 'Fz',
              'type': 'scatter',
              'uid': 'db080ca8-ecc8-47ad-90a5-034052c37429',
              'x': {'bdata': ('mpmZmZmZyb9KDAIrhxbJv/p+arx0k8' ... 'Q73/8/Vg4tsp3v/z8AAAAAAAAAQA=='),
                    'dtype': 'f8'},
              'y': {'bdata': ('ZvfkYaHW9L4aUdobfGHyvkhQ/Bhz19' ... '3gLeA+jSjtDb4w6T7V52or9pftPg=='),
                    'dtype': 'f8'}},
             {'mode': 'lines',
              'name': 'FC3',
              'type': 'scatter',
              'uid': 'e438e8a3-5065-427e-9b60-5252cd0ab3ea',
              'x': {'bdata': ('mpmZmZmZyb9KDAIrhxbJv/p+arx0k8' ... 'Q73/8/Vg4tsp3v/z8AAAAAAAAAQA=='),
                    'dtype': 'f8'},
              'y': {'bdata': ('UPwYc9cS8r7/If32deDsvixDHOviNr' ... 'JNYrA+QILix5i71j79ZffkYaHmPg=='),
                    'dtype': 'f8'}},
             {'mode': 'lines',
              'name': 'FC1',
              'type': 'scatter',
              'uid': '3a1b654e-